# Phase 02B: Tích hợp VLM Open-Weights trên Google Colab GPU

Notebook này thực hiện tích hợp mô hình Vision-Language Model (`Qwen/Qwen2.5-VL-3B-Instruct` hoặc `7B`) với **Photo Advisor Pipeline**.
- Sử dụng **cùng một đường Provider & Pipeline** (`photo_advisor.vlm_adapter`, `photo_advisor.cli`).
- Tự động nhận diện tài nguyên GPU và áp dụng cấu hình precision / quantization / image limits đồng bộ.
- Truyền ảnh đã chuẩn hóa upright sRGB và bảo toàn hệ tọa độ crop của Engine 02A.
- Toàn bộ quy trình từ inference VLM đến xuất bản phẩm khép kín trong **duy nhất 1 lần gọi** `run_pipeline`.
- Có rào chắn lỗi đa giai đoạn (OOM, timeout, preprocess, decode) và kiểm chứng schema nghiêm ngặt.
- Chế độ rõ ràng: `EXECUTION_MODE='mock'` (offline, an toàn) và `EXECUTION_MODE='real'` (GPU remote, bắt buộc ảnh thật).

In [ ]:
# [Cell 1] Kiểm tra tài nguyên phần cứng: GPU / CUDA / VRAM / System RAM
import sys
import psutil
import torch

print(f"Python executable: {sys.executable}")
print(f"Python version:    {sys.version.split()[0]}")
print(f"PyTorch version:   {torch.__version__}")

ram = psutil.virtual_memory()
print(f"System RAM:        {ram.total / (1024**3):.2f} GB (Khả dụng: {ram.available / (1024**3):.2f} GB)")

cuda_available = torch.cuda.is_available()
print(f"CUDA Available:    {cuda_available}")

if cuda_available:
    gpu_count = torch.cuda.device_count()
    current_device = torch.cuda.current_device()
    device_name = torch.cuda.get_device_name(current_device)
    vram_bytes = torch.cuda.get_device_properties(current_device).total_memory
    vram_gb = vram_bytes / (1024**3)
    print(f"Số lượng GPU:      {gpu_count}")
    print(f"Thiết bị chính:    [{current_device}] {device_name}")
    print(f"Tổng VRAM:         {vram_gb:.2f} GB")
    print(f"CUDA Capability:   {torch.cuda.get_device_capability(current_device)}")
else:
    print("\n[CHÚ Ý / CẢNH BÁO]: Runtime hiện tại không có GPU CUDA.")
    print("Để chạy VLM thật, hãy vào: Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4/L4/A100).")
    print("Ở chế độ không có GPU, notebook sẽ tự động kích hoạt MockVLMProvider để kiểm chứng toàn bộ luồng.")


In [ ]:
# [Cell 2] Cài đặt pinned dependencies hỗ trợ Qwen2.5-VL và Photo Advisor Pipeline
# Ghi chú rationale chọn phiên bản:
# - transformers>=4.45.0,<5.0.0: Hỗ trợ chính thức kiến trúc Qwen2.5-VL; chặn major v5 để tránh breaking changes API.
# - accelerate>=0.26.0,<1.0.0: Hỗ trợ tự động phân bổ device map, quản lý bfloat16 và offload bộ nhớ GPU/CPU.
# - Pillow==11.3.0 & numpy==2.2.6: Đồng bộ chính xác với môi trường cục bộ và test suite Phase 02A/02B.
# - jsonschema==4.26.0: Xác thực schema output authoritative Draft 7 không gây ngoại lệ unhashable/crash.
# - qwen-vl-utils>=0.0.8: Tiện ích xử lý kích thước ảnh thông minh theo khuyến nghị chính thức của Qwen-VL team.
# - bitsandbytes>=0.43.0: Dự phòng lượng tử hóa NF4/INT8 nếu chạy trên GPU VRAM thấp (Colab T4).
!pip install -q --upgrade pip
!pip install -q "transformers>=4.45.0,<5.0.0" "accelerate>=0.26.0,<1.0.0" "Pillow==11.3.0" "numpy==2.2.6" "jsonschema==4.26.0" "qwen-vl-utils>=0.0.8" "bitsandbytes>=0.43.0" "torchvision"

import transformers
import PIL
import numpy as np
import jsonschema
print(f"✓ Dependencies đã cài đặt: transformers={transformers.__version__}, Pillow={PIL.__version__}, numpy={np.__version__}, jsonschema={jsonschema.__version__}")


In [ ]:
# [Cell 3] Nạp mã nguồn dự án Photo Advisor và kiểm tra Prerequisites (dừng ngay nếu thiếu)
import os
import sys

repo_root = os.path.abspath(".")
src_path = os.path.join(repo_root, "src")

# Nếu đang trong thư mục notebooks hoặc thư mục con khác
if not os.path.exists(os.path.join(src_path, "photo_advisor")):
    parent_src = os.path.abspath(os.path.join(repo_root, "..", "src"))
    if os.path.exists(os.path.join(parent_src, "photo_advisor")):
        src_path = parent_src
        repo_root = os.path.abspath(os.path.join(repo_root, ".."))

if not os.path.exists(os.path.join(src_path, "photo_advisor")):
    raise FileNotFoundError(
        f"[PREREQUISITE THẤT BẠI]: Không tìm thấy thư mục 'src/photo_advisor' tại '{src_path}'.\n"
        "Vui lòng đảm bảo bạn đã clone hoặc upload toàn bộ mã nguồn dự án vào môi trường làm việc."
    )

if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Import trực tiếp các module chính thức của dự án
import photo_advisor
from photo_advisor.cli import run_pipeline, create_synthetic_test_image
from photo_advisor.vlm_adapter import (
    LocalTransformersProvider,
    MockVLMProvider,
    ReplayVLMProvider,
    VLMAdapter,
    VLMRequest,
    VLMResponse,
    validate_vlm_schema,
)
from photo_advisor.processor import compute_sha256, normalize_input_image, process_image
from photo_advisor.validator import validate_and_clamp_recipe

print(f"✓ Nạp thành công photo_advisor từ: {photo_advisor.__file__}")


In [ ]:
# [Cell 4] Cấu hình chế độ thực thi và tài nguyên (E1)
# - EXECUTION_MODE = 'mock': Luôn dùng MockVLMProvider (an toàn 100%, không tải weights kể cả khi có CUDA).
# - EXECUTION_MODE = 'real': Yêu cầu có GPU CUDA và ảnh chân dung thật do người dùng chỉ định.
EXECUTION_MODE = "mock"  # Đổi thành 'real' khi chạy smoke test trên phiên Google Colab GPU thật

import torch

if torch.cuda.is_available() and EXECUTION_MODE == "real":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Phát hiện GPU CUDA, VRAM: {vram_gb:.2f} GB")
    
    # Cấu hình tài nguyên thực tế
    if vram_gb < 16.0:
        # Colab T4 (~15GB VRAM): Khuyến nghị Qwen2.5-VL-3B-Instruct (qwen-research license)
        MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
        COMPUTE_DTYPE = torch.float16 if not torch.cuda.is_bf16_supported() else torch.bfloat16
        USE_4BIT = False
    else:
        # Colab L4 (24GB) hoặc A100 (40GB/80GB): Qwen2.5-VL-7B-Instruct (Apache-2.0 license)
        MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
        COMPUTE_DTYPE = torch.bfloat16
        USE_4BIT = False
    DEVICE = "cuda"
else:
    if EXECUTION_MODE == "mock":
        print("[EXECUTION_MODE='mock']: Kích hoạt MockVLMProvider deterministic (an toàn, không tải weights).")
    else:
        print("[CẢNH BÁO]: Không có GPU CUDA; tự động chuyển sang mock mode để kiểm chứng offline.")
    MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
    COMPUTE_DTYPE = None
    USE_4BIT = False
    DEVICE = "cpu"

# Giới hạn kích thước ảnh & tokens nhằm đảm bảo an toàn bộ nhớ VRAM
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1280 * 28 * 28
MAX_NEW_TOKENS = 512
TIME_BUDGET_SECONDS = 60.0

print(f"Cấu hình: Mode={EXECUTION_MODE}, Model={MODEL_ID}, Device={DEVICE}, dtype={COMPUTE_DTYPE}, 4bit={USE_4BIT}")
print(f"Giới hạn ảnh: {MIN_PIXELS} .. {MAX_PIXELS} px | Max tokens: {MAX_NEW_TOKENS} | Budget: {TIME_BUDGET_SECONDS}s")


In [ ]:
# [Cell 5] Khởi tạo VLM Provider theo EXECUTION_MODE (E1)
import time

t0 = time.perf_counter()
if EXECUTION_MODE == "real":
    if not torch.cuda.is_available():
        raise RuntimeError(
            "[LỖI CẤU HÌNH - E1]: EXECUTION_MODE='real' yêu cầu môi trường GPU CUDA. "
            "Vui lòng vào Runtime -> Change runtime type -> Hardware accelerator -> T4/L4 GPU, "
            "hoặc chuyển về EXECUTION_MODE='mock' để chạy thử offline."
        )
    print(f"Khởi tạo LocalTransformersProvider ({MODEL_ID}) trên {DEVICE} ...")
    provider = LocalTransformersProvider(
        model_id=MODEL_ID,
        device=DEVICE,
        load_in_4bit=USE_4BIT,
        compute_dtype=COMPUTE_DTYPE,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
        max_new_tokens=MAX_NEW_TOKENS,
        time_budget_seconds=TIME_BUDGET_SECONDS,
    )
    print(f"✓ Provider đã khởi tạo với cấu hình: {provider.get_effective_config()}")
else:
    # Luôn dùng MockVLMProvider khi EXECUTION_MODE == 'mock', kể cả khi máy có GPU CUDA
    print(f"Khởi tạo MockVLMProvider (EXECUTION_MODE='{EXECUTION_MODE}' - offline mock / debug mode)...")
    provider = MockVLMProvider(simulation_mode="normal")
    print("✓ Khởi tạo MockVLMProvider thành công (an toàn, không tải weights).")


In [ ]:
# [Cell 6] Chuẩn bị ảnh chân dung đầu vào và xác định Goal
# QUY TẮC NGUỒN ẢNH (E1):
# - Chế độ 'real': BẮT BUỘC người dùng chỉ định ảnh người thật; dừng ngay nếu thiếu, KHÔNG tự ý sinh synthetic!
# - Chế độ 'mock': Dùng ảnh có sẵn hoặc sinh synthetic để kiểm chứng offline.
import io
import os
from PIL import Image

# Người dùng upload ảnh thật lên Colab và chỉ định đường dẫn dưới đây:
input_image_path = "work/phase02b/sample_real_portrait.jpg"
input_source_desc = "real_user_photo: Human portrait reference (explicit consent verified)"
GOAL = "natural_portrait"  # Lựa chọn: natural_portrait | soft_glow | cinematic | silhouette

if EXECUTION_MODE == "real":
    if not os.path.exists(input_image_path):
        raise FileNotFoundError(
            f"[DỪNG TRƯỚC INFERENCE - E1]: EXECUTION_MODE='real' yêu cầu ảnh chân dung người thật tại '{input_image_path}'. "
            "Hệ thống từ chối tự động sinh ảnh synthetic khi chạy với real provider. "
            "Vui lòng upload ảnh người thật lên môi trường Colab hoặc đổi EXECUTION_MODE='mock' để kiểm thử offline."
        )
    with open(input_image_path, "rb") as f:
        raw_bytes = f.read()
    print(f"✓ Sử dụng ảnh chân dung người thật: {input_image_path} ({len(raw_bytes):,} bytes)")
else:
    # Mock mode
    if os.path.exists(input_image_path):
        with open(input_image_path, "rb") as f:
            raw_bytes = f.read()
        print(f"[MOCK MODE]: Sử dụng ảnh có sẵn: {input_image_path} ({len(raw_bytes):,} bytes)")
    else:
        print("[MOCK MODE]: Không có ảnh mẫu; sinh ảnh tổng hợp (synthetic) chuẩn để kiểm chứng logic...")
        input_image_path = "work/phase02b/synthetic_test_portrait.png"
        input_source_desc = "synthetic"
        os.makedirs(os.path.dirname(input_image_path), exist_ok=True)
        raw_bytes = create_synthetic_test_image(800, 600, style="portrait")
        with open(input_image_path, "wb") as f:
            f.write(raw_bytes)

print(f"Đường dẫn tệp:           {input_image_path}")
print(f"Nguồn gốc ảnh:           {input_source_desc}")
print(f"Kích thước file:         {len(raw_bytes):,} bytes")
print(f"Mục tiêu (Goal):          {GOAL}")


In [ ]:
# [Cell 7] Xác nhận cấu hình và trạng thái sẵn sàng (KHÔNG gọi model tại đây)
# LƯU Ý QUAN TRỌNG (D1):
# Loại bỏ hoàn toàn bước gọi riêng VLMAdapter.process tại đây.
# Toàn bộ quy trình inference và xuất bản phẩm được gom vào DUY NHẤT một lần gọi
# run_pipeline() ở Cell 8 tiếp theo để tránh trùng lặp.
print(f"Chế độ thực thi:        {EXECUTION_MODE}")
print(f"Provider đã sẵn sàng:   {provider.__class__.__name__}")
print(f"Ảnh đầu vào:             {input_image_path} ({len(raw_bytes):,} bytes)")
print(f"Nguồn gốc ảnh:           {input_source_desc}")
print(f"Mục tiêu xử lý (Goal):   {GOAL}")


In [ ]:
# [Cell 8] Chạy toàn trình Pipeline (DUY NHẤT 1 LỜI GỌI MODEL) và xuất bản phẩm
# Khắc phục ghi đè và hỗ trợ smoke test nhiều ảnh (E2):
# Mỗi lượt chạy tự động sinh thư mục riêng biệt kèm Run ID / UUID mới, bảo vệ nguyên vẹn các kết quả cũ.
import datetime
import json
import os
import uuid

output_dir = "work/phase02b/run_output_colab"
# Tự động gán run ID / UUID để mỗi lượt chạy sinh thư mục mới độc lập, không ghi đè (E2)
if output_dir == "work/phase02b/run_output_colab":
    run_uid = f"run_{datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"
    output_dir = f"{output_dir}_{run_uid}"

os.makedirs(output_dir, exist_ok=True)

print(f"Bắt đầu thực thi Pipeline duy nhất vào thư mục: {output_dir} ...")
manifest = run_pipeline(
    image_bytes=raw_bytes,
    goal=GOAL,
    output_dir=output_dir,
    image_id=os.path.splitext(os.path.basename(input_image_path))[0],
    input_file_path=input_image_path,
    input_source=input_source_desc,
    provider=provider,
    prompt_version="prompt_v1",
    model_id=MODEL_ID,
)

print(f"\n✓ Pipeline hoàn tất! Run ID: {manifest['metadata']['run_id']}")
print(f"  Tổng thời gian: {manifest['metadata']['total_runtime_ms']} ms")
if "timings_ms" in manifest["metadata"]:
    t_info = manifest["metadata"]["timings_ms"]
    print(f"  - Model Load:       {t_info.get('model_load_ms', 0.0)} ms")
    print(f"  - Model Generation: {t_info.get('model_generation_ms', 0.0)} ms")
    print(f"  - Pipeline Total:   {t_info.get('pipeline_total_ms', 0.0)} ms")

# Kiểm tra kết quả xuất bản phẩm và Envelope truy xuất nguồn gốc từ đĩa
print("\n=== DANH SÁCH BẢN PHẨM TRONG THƯ MỤC RUN ===")
for fname in sorted(os.listdir(output_dir)):
    fsize = os.path.getsize(os.path.join(output_dir, fname))
    print(f"  - {fname:<25} ({fsize:,} bytes)")

raw_resp_path = os.path.join(output_dir, "vlm_raw_response.json")
if os.path.exists(raw_resp_path):
    with open(raw_resp_path, "r", encoding="utf-8") as f:
        vlm_raw_data = json.load(f)
    env = vlm_raw_data.get("envelope", {})
    print("\n=== VLM PROVENANCE ENVELOPE ===")
    print(f"  Model ID:        {env.get('model_id')}")
    print(f"  Model Revision:  {env.get('model_revision')}")
    print(f"  Provider Type:   {env.get('provider_type')}")
    print(f"  Is Real Model:   {env.get('is_real_model')}")
    print(f"  Status:          {env.get('status')}")
    print(f"  Schema SHA-256:  {env.get('schema_sha256', '')[:16]}...")
    print(f"  Prompt SHA-256:  {env.get('prompt_sha256', '')[:16]}...")
    print(f"  Timings (ms):    {env.get('timings_ms')}")
    print(f"  Effective Cfg:   {env.get('effective_config')}")

print("\n=== CÁC PHƯƠNG ÁN CANDIDATE ===")
for cand in manifest["candidates"]:
    print(f"  [{cand['status'].upper()}] {cand['candidate_id']}: {cand['source_name']}")
    print(f"     Preview: {cand['preview_filename']} | Ghi chú: {cand['notes']}")


### Hướng dẫn Smoke Test Batch 2 Ảnh Người Thật (E2)

Khi mở phiên Google Colab GPU thực tế, thực hiện smoke test 2 ảnh chân dung thật theo quy trình:

1. **Chuẩn bị 2 ảnh người thật:**
   - Upload 2 ảnh vào `work/phase02b/` (ví dụ: `sample_real_portrait_1.jpg` và `sample_real_portrait_2.jpg`).
   - Đảm bảo người trong ảnh đã có sự đồng thuận rõ ràng (explicit consent).

2. **Thực thi tuần tự từng ảnh:**
   - **Ảnh 1:** Đặt `input_image_path = 'work/phase02b/sample_real_portrait_1.jpg'` và `GOAL = 'natural_portrait'` tại Cell 6, chạy lại Cell 6 để nạp ảnh, rồi chạy Cell 8.
     Kết quả được lưu vào thư mục riêng: `work/phase02b/run_output_colab_run_<time>_<uuid>/`.
   - **Ảnh 2:** Đổi `input_image_path = 'work/phase02b/sample_real_portrait_2.jpg'` và `GOAL = 'soft_glow'` tại Cell 6, chạy lại Cell 6 để nạp ảnh mới, rồi chạy Cell 8.
     Kết quả được lưu vào thư mục MỚI, hoàn toàn không ghi đè hoặc xóa output của Ảnh 1.

3. **Hoặc chạy batch 2 ảnh bằng snippet sau:**
```python
# Snippet chạy kiểm chứng 2 ảnh thật liên tiếp trong 1 thao tác:
smoke_batch = [
    {"path": "work/phase02b/sample_real_portrait_1.jpg", "goal": "natural_portrait"},
    {"path": "work/phase02b/sample_real_portrait_2.jpg", "goal": "soft_glow"},
]
for item in smoke_batch:
    uid = f"run_{datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"
    run_out = f"work/phase02b/run_output_colab_{uid}"
    with open(item["path"], "rb") as f: data = f.read()
    run_pipeline(
        image_bytes=data,
        goal=item["goal"],
        output_dir=run_out,
        input_file_path=item["path"],
        input_source="real_user_photo: Human portrait reference (explicit consent verified)",
        provider=provider,
    )
```


In [ ]:
# [Cell 9] Chạy toàn bộ Test Suite của dự án (ép CUDA_VISIBLE_DEVICES="" để kiểm thử offline an toàn tuyệt đối)
!CUDA_VISIBLE_DEVICES="" python3 -m unittest discover -s tests -p "test_*.py" -v


In [ ]:
# [Cell 10] Đóng gói kết quả (Zip) và hướng dẫn tải về workspace local
import shutil

zip_filename = "phase02b_colab_artifacts"
shutil.make_archive(zip_filename, "zip", output_dir)
print(f"✓ Đã nén kết quả của lượt chạy ({output_dir}) thành công: {zip_filename}.zip")

print("\n" + "="*60)
print("HƯỚNG DẪN TẢI KẾT QUẢ VỀ WORKSPACE:")
print("1. Tải file zip về máy tính:")
print("   from google.colab import files; files.download('phase02b_colab_artifacts.zip')")
print("2. Giải nén vào thư mục dự án:")
print(f"   unzip phase02b_colab_artifacts.zip -d {output_dir}/")
print("="*60)

try:
    from google.colab import files
    files.download(f"{zip_filename}.zip")
    print("Đang mở hộp thoại tải xuống...")
except ImportError:
    print("(Môi trường ngoài Colab: file zip đã sẵn sàng tại thư mục hiện tại)")
